# 02 · Where fine-tuning fits in

Order of escalation (cheapest first):

```
Prompt engineering → Few-shot examples → RAG → Fine-tuning (LoRA/QLoRA) → Full fine-tuning → Pretraining
```

| Need | Best tool |
|---|---|
| Better instructions / format | Prompting |
| Up-to-date or private **knowledge** | RAG |
| Specific **behaviour, style, format** | Fine-tuning |
| Both knowledge + behaviour | RAG + fine-tuned model |

## Fine-tuning flavours
| Method | What is trained | Memory | When |
|---|---|---|---|
| Full FT | all weights | huge (~16 bytes/param w/ Adam) | lots of GPUs |
| **LoRA** | low-rank adapters, base in 16-bit | medium | one 24 GB+ GPU |
| **QLoRA** (our choice) | adapters, base in **4-bit NF4** | small | free T4 16 GB |

In [ ]:
# CPU-only: rough GPU-memory estimator (weights + optimizer/grad state, ignores activations)
N = 3.21e9            # Llama-3.2-3B params
def gb(x): return x/1e9

full  = N*2 + N*2 + N*8          # bf16 weights + bf16 grads + fp32 Adam (2 states)
lora_trainable = 0.03*N          # generous upper bound for LoRA params
lora  = N*2 + lora_trainable*(2+8)
qlora = N*0.5 + lora_trainable*(2+8)   # 4-bit ≈ 0.5 byte/param

print(f"Full fine-tune : ~{gb(full):5.1f} GB")
print(f"LoRA (16-bit)  : ~{gb(lora):5.1f} GB")
print(f"QLoRA (4-bit)  : ~{gb(qlora):5.1f} GB  (+ activations; fits a 16 GB T4 with short seqs)")

In [ ]:
def recommend(needs_new_facts: bool, needs_style_or_format: bool, have_examples: int, gpu_gb: int):
    out = []
    if needs_new_facts: out.append("RAG for the knowledge")
    if needs_style_or_format:
        if have_examples < 200: out.append("Few-shot prompting first (too few examples to fine-tune)")
        else: out.append("QLoRA fine-tune" if gpu_gb < 24 else "LoRA fine-tune")
    return out or ["Prompt engineering is probably enough"]

print(recommend(False, True, 5000, 16))
print(recommend(True, False, 0, 16))

## Our pipeline

```
01-02 concepts → 03 instruction format → 04 data prep → 05 train (QLoRA) → 06 evaluate/export
```

Tools: **Unsloth** (fast, memory-efficient kernels), **TRL** `SFTTrainer`, **PEFT**, **bitsandbytes**, 🤗 **Datasets**.

Next → `03_Instruction_tuning.ipynb`